In [1]:
import numpy as np
import pandas as pd
from datetime import datetime, date, timedelta
from dateutil.relativedelta import relativedelta
import pytz
import yfinance as yf
import pyodbc
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
import time
import logging
import pandas as pd
from truedata import TD_hist
import requests
import sys, os
sys.path.insert(0, r"C:\Users\anike\Desktop\Ocean_dev\Momentum Handover\Momentum Handover\MOMENTUM_DB_2")
from truedata_connector import get_td_obj


In [2]:
def fetch_truedata_history(
    ticker_list: list,
    duration: str = '1 Y',
    bar_size: str = 'EOD',
    sleep_time: float = 0.1,
    max_retries: int = 5          # ← new: retries per ticker on IP/session drop
) -> tuple[pd.DataFrame, list]:
    """
    Fetches historical data from TrueData with auto-reconnect on IP/session drops.
    Drop-in replacement for the original fetch_truedata_history.
    """
    logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

    td_hist = get_td_obj()        # reuses existing session, reconnects if dropped
    df_list = []
    error_list = []

    for ticker in ticker_list:
        fetched = False
        for attempt in range(1, max_retries + 1):
            try:
                df = td_hist.get_historic_data([ticker], duration=duration, bar_size=bar_size)

                if df is None or (hasattr(df, 'empty') and df.empty):
                    logging.warning(f'No data fetched for {ticker}')
                    error_list.append(ticker)
                    fetched = True   # don't retry — genuinely no data
                    break

                df['Ticker'] = ticker

                # Column renaming (same as original)
                rename_dict = {}
                if 'timestamp' in df.columns:
                    rename_dict['timestamp'] = 'Date'
                elif 'datetime' in df.columns:
                    rename_dict['datetime'] = 'Date'
                elif 'date' in df.columns:
                    rename_dict['date'] = 'Date'
                rename_dict.update({
                    'high': 'High', 'low': 'Low',
                    'close': 'Close', 'open': 'Open'
                })
                df = df.rename(columns=rename_dict)

                df_list.append(df)
                logging.info(f"Fetched data for {ticker} ({len(df)} rows).")
                fetched = True
                break   # success

            except Exception as e:
                err_str = str(e).lower()
                is_conn = any(k in err_str for k in [
                    'connection', 'timeout', 'reset', 'broken', 'eof',
                    'socket', 'session', 'disconnect', 'auth'
                ])

                if attempt < max_retries:
                    wait = 2 ** attempt
                    logging.warning(f"[{ticker}] Attempt {attempt}/{max_retries} failed: {e}. "
                                    f"Reconnecting in {wait}s...")
                    time.sleep(wait)
                    try:
                        td_hist = get_td_obj(force_reconnect=True)  # fresh session
                    except Exception as ce:
                        logging.error(f"Reconnect failed: {ce}")
                else:
                    logging.error(f"All {max_retries} attempts failed for {ticker}: {e}")

        if not fetched:
            error_list.append(ticker)

        time.sleep(sleep_time)

    final_df = pd.concat(df_list, ignore_index=True) if df_list else pd.DataFrame()
    return final_df, error_list


In [3]:
def process_portfolio(nav_df, ticker_data, initial_value=75, inception_date=None, output_file=None):
    """
    Process portfolio allocation with month-by-month rebalancing.
    """
    df_lis = []
    last_month_value = {}
    last_month_quantity = {}

    nav_df = nav_df.sort_values(['Date', 'Ticker']).copy()
    nav_df['Date'] = pd.to_datetime(nav_df['Date'])
    ticker_data = ticker_data.sort_values(['Ticker', 'Date']).copy()
    ticker_data['Date'] = pd.to_datetime(ticker_data['Date'])

    if inception_date is None:
        inception_date = nav_df['Date'].min()
    else:
        inception_date = pd.to_datetime(inception_date)

    for year_month in nav_df['Year-Month'].drop_duplicates():
        month_nav = nav_df[nav_df['Year-Month'] == year_month].copy()
        tickers = month_nav['Ticker'].dropna().unique().tolist()
        selection_date = pd.to_datetime(month_nav['Date'].min())
        year_month_date = pd.to_datetime(f"{year_month}-01")

        prev_month_start = year_month_date - relativedelta(months=2)
        curr_month_start = year_month_date
        curr_month_end = year_month_date + pd.offsets.MonthEnd(0)

        stock_data = ticker_data[
            (ticker_data['Date'] >= prev_month_start)
            & (ticker_data['Date'] <= curr_month_end)
            & (ticker_data['Ticker'].isin(tickers))
        ].copy()
        stock_data = stock_data[stock_data['Date'] >= inception_date].copy()
        stock_data['%change'] = stock_data.groupby('Ticker')['Close'].pct_change().fillna(stock_data['Close'] / stock_data['Open'] - 1)

        stock_data_flt = stock_data[
            (stock_data['Date'] >= curr_month_start) & (stock_data['Date'] <= curr_month_end)
        ].copy()
        if stock_data_flt.empty:
            continue

        if not last_month_value:
            allocation_per_stock = initial_value / len(tickers)
            stock_allocations = {ticker: allocation_per_stock for ticker in tickers}
        else:
            stock_allocations = {ticker: last_month_value[ticker] for ticker in tickers if ticker in last_month_value}
            dropped_stocks = [ticker for ticker in last_month_value if ticker not in tickers]
            dropped_value = sum(last_month_value[ticker] for ticker in dropped_stocks)
            new_stocks = [ticker for ticker in tickers if ticker not in last_month_value]
            if new_stocks:
                allocation_per_stock = dropped_value / len(new_stocks) if dropped_value else 0.0
                for ticker in new_stocks:
                    stock_allocations[ticker] = allocation_per_stock
                    
                    # New stock: bought at Open on Day 1, so %change = (Close/Open) - 1
                    ticker_idx = stock_data_flt[stock_data_flt['Ticker'] == ticker].index
                    if not ticker_idx.empty:
                        f_idx = ticker_idx[0]
                        open_px = stock_data_flt.loc[f_idx, 'Open']
                        close_px = stock_data_flt.loc[f_idx, 'Close']
                        if open_px and open_px > 0:
                            stock_data_flt.loc[f_idx, '%change'] = (close_px / open_px) - 1

        for ticker, init_value in stock_allocations.items():
            ticker_index = stock_data_flt[stock_data_flt['Ticker'] == ticker].index
            ticker_df = stock_data_flt.loc[ticker_index].copy()
            if ticker_df.empty:
                continue

            stock_data_flt.loc[ticker_index, 'Initial_Allocation'] = init_value
            stock_data_flt.loc[ticker_index, 'Selection_Date'] = selection_date
            stock_data_flt.loc[ticker_index, 'Buy_Hold_Value'] = init_value * (
                (1 + stock_data_flt.loc[ticker_index, '%change'].fillna(0)).cumprod()
            )

            buy_price = float(ticker_df.iloc[0]['Close'])
            quantity = last_month_quantity.get(ticker, init_value / buy_price if buy_price else 0.0)
            stock_data_flt.loc[ticker_index, 'Buy_Price'] = buy_price
            stock_data_flt.loc[ticker_index, 'Quantity'] = quantity
            if 'Real_Rank' in month_nav.columns:
                stock_data_flt.loc[ticker_index, 'Real_Rank'] = month_nav.loc[month_nav['Ticker'] == ticker, 'Real_Rank'].iloc[0]

        last_month_quantity = stock_data_flt.groupby('Ticker')['Quantity'].last().to_dict()
        last_month_value = stock_data_flt.groupby('Ticker')['Buy_Hold_Value'].last().to_dict()
        stock_data_flt['Total_Portfolio_Value'] = stock_data_flt.groupby('Date')['Buy_Hold_Value'].transform('sum')
        df_lis.append(stock_data_flt)

    final_df = pd.concat(df_lis, ignore_index=True).sort_values(['Date', 'Ticker']).reset_index(drop=True)

    if output_file:
        final_df.to_excel(output_file, index=False)

    return final_df


def build_weighted_hedge_segment(hedge_prices, start_date, end_date, base_values, segment_name, new_tickers=None):
    """
    new_tickers: list of tickers entering this segment fresh (bought at Open on Day 1).
                 All others are assumed to be continuing (overnight close-to-close on Day 1).
    """
    if new_tickers is None:
        new_tickers = []

    segment = hedge_prices[
        (hedge_prices['Date'] >= start_date)
        & (hedge_prices['Date'] <= end_date)
        & (hedge_prices['Ticker'].isin(base_values))
    ][['Date', 'Ticker', 'Open', 'Close']].copy()
    if segment.empty:
        return segment

    segment = segment.sort_values(['Ticker', 'Date'])

    # Compute close-to-close using full hedge_prices history (not just segment window)
    # so continuing hedges get the correct overnight return on Day 1
    full_history = hedge_prices[
        hedge_prices['Ticker'].isin(base_values)
    ][['Date', 'Ticker', 'Close']].sort_values(['Ticker', 'Date']).copy()
    full_history['%change_full'] = full_history.groupby('Ticker')['Close'].pct_change()

    segment = segment.merge(
        full_history[['Date', 'Ticker', '%change_full']],
        on=['Date', 'Ticker'],
        how='left'
    )

    # Default: use close-to-close from full history
    segment['%change'] = segment['%change_full']

    # For new tickers on Day 1: override to intraday (bought at Open)
    for ticker in new_tickers:
        mask = (segment['Ticker'] == ticker) & (segment['Date'] == start_date)
        if mask.any():
            segment.loc[mask, '%change'] = (segment.loc[mask, 'Close'] / segment.loc[mask, 'Open']) - 1

    # For any remaining NaN on Day 1 (truly first trading day ever): fallback to intraday
    day1_mask = segment['Date'] == start_date
    segment.loc[day1_mask & segment['%change'].isna(), '%change'] = (
        segment.loc[day1_mask & segment['%change'].isna(), 'Close'] /
        segment.loc[day1_mask & segment['%change'].isna(), 'Open'] - 1
    )

    segment = segment.drop(columns=['%change_full'])
    segment['Initial_Allocation'] = segment['Ticker'].map(base_values)
    segment['ret_factor'] = 1 + segment['%change'].fillna(0)
    segment['cum_factor'] = segment.groupby('Ticker')['ret_factor'].cumprod()
    segment['Buy_Hold_Value'] = segment['Initial_Allocation'] * segment['cum_factor']

    buy_prices = segment.groupby('Ticker')['Close'].transform('first')
    segment['Buy_Price'] = buy_prices
    segment['Quantity'] = np.where(buy_prices > 0, segment['Initial_Allocation'] / buy_prices, 0.0)
    segment['Selection_Date'] = start_date
    segment['Hedge_Segment'] = segment_name
    segment['Total_Portfolio_Value'] = segment.groupby('Date')['Buy_Hold_Value'].transform('sum')
    return segment.drop(columns=['ret_factor', 'cum_factor'])


def build_rebalanced_hedge_book(base_portfolio_df, portfolio_end_date=None, default_hedge_value=25.0):
    if portfolio_end_date is None:
        portfolio_end_date = pd.to_datetime(base_portfolio_df['Date']).max()
    else:
        portfolio_end_date = pd.to_datetime(portfolio_end_date)

    cutoff_date = pd.Timestamp('2025-11-30')
    if portfolio_end_date <= cutoff_date:
        return pd.DataFrame()

    hedge_start_factor = base_portfolio_df[
        (base_portfolio_df['Ticker'] == 'GOLDBEES') & (base_portfolio_df['Date'] <= cutoff_date)
    ].sort_values('Date')
    hedge_seed = float(hedge_start_factor['Buy_Hold_Value'].iloc[-1]) if not hedge_start_factor.empty else default_hedge_value

    # fetch both CPSEETF and PHARMABEES so we can swap CPSEETF -> PHARMABEES in June
    hedge_prices = fetch_truedata_history(
        ticker_list=['GOLDBEES', 'SILVERBEES', 'MOGSEC', 'LIQUIDCASE', 'CPSEETF', 'PHARMABEES', 'NEXT50IETF'],
        duration='5 Y',
        bar_size='EOD',
        sleep_time=0.1
    )[0]

    segments = []

    # December->January segment
    decjan_start = pd.Timestamp('2025-12-01')
    decjan_end = min(pd.Timestamp('2026-01-31'), portfolio_end_date)
    if portfolio_end_date >= decjan_start:
        decjan_values = {'GOLDBEES': 0.60 * hedge_seed, 'SILVERBEES': 0.20 * hedge_seed, 'MOGSEC': 0.20 * hedge_seed}
        df_decjan = build_weighted_hedge_segment(
            hedge_prices,
            start_date=decjan_start,
            end_date=decjan_end,
            base_values=decjan_values,
            segment_name='2025-12_to_2026-01'
        )
        if not df_decjan.empty:
            segments.append(df_decjan)
        else:
            df_decjan = pd.DataFrame()
    else:
        df_decjan = pd.DataFrame()

    # February segment (uses last DecJan snapshot to size)
    feb_start = pd.Timestamp('2026-02-01')
    feb_end = min(pd.Timestamp('2026-02-28'), portfolio_end_date)
    if portfolio_end_date >= feb_start and not df_decjan.empty:
        feb_factor = df_decjan.groupby('Date', as_index=False)['Buy_Hold_Value'].sum().sort_values('Date')['Buy_Hold_Value'].iloc[-1]
        feb_values = {'GOLDBEES': 0.40 * feb_factor, 'MOGSEC': 0.60 * feb_factor}
        df_feb = build_weighted_hedge_segment(
            hedge_prices,
            start_date=feb_start,
            end_date=feb_end,
            base_values=feb_values,
            segment_name='2026-02'
        )
        if not df_feb.empty:
            segments.append(df_feb)
        else:
            df_feb = pd.DataFrame()
    else:
        df_feb = pd.DataFrame()

    # March->April segment based on Feb snapshot
    mar_start = pd.Timestamp('2026-03-01')
    mar_end = min(pd.Timestamp('2026-04-30'), portfolio_end_date)
    df_mar = pd.DataFrame()
    if portfolio_end_date >= mar_start and not df_feb.empty:
        feb_last_date = df_feb['Date'].max()
        feb_last = df_feb[df_feb['Date'] == feb_last_date].set_index('Ticker')['Buy_Hold_Value'].to_dict()
        mar_values = {'GOLDBEES': feb_last.get('GOLDBEES', 0.0), 'LIQUIDCASE': feb_last.get('MOGSEC', 0.0)}
        df_mar = build_weighted_hedge_segment(
            hedge_prices,
            start_date=mar_start,
            end_date=mar_end,
            base_values=mar_values,
            segment_name='2026-03_to_2026-04' if portfolio_end_date >= mar_end else '2026-03_onward',
            new_tickers=['LIQUIDCASE']  # LIQUIDCASE bought fresh at Open on Mar 1 (replaces MOGSEC)
        )
        if not df_mar.empty:
            segments.append(df_mar)

    # May segment: include CPSEETF so we can sell it at June
    may_start = pd.Timestamp('2026-05-01')
    may_end = min(pd.Timestamp('2026-05-31'), portfolio_end_date)
    df_may = pd.DataFrame()
    if portfolio_end_date >= may_start and not df_mar.empty:
        mar_last_date = df_mar['Date'].max()
        mar_last = df_mar[df_mar['Date'] == mar_last_date].set_index('Ticker')['Buy_Hold_Value'].to_dict()
        total_hedge_value = sum(mar_last.values())

        may_values = {
            'GOLDBEES': 0.20 * total_hedge_value,
            'LIQUIDCASE': 0.40 * total_hedge_value,
            'CPSEETF': 0.20 * total_hedge_value,
            'NEXT50IETF': 0.20 * total_hedge_value,
        }
        df_may = build_weighted_hedge_segment(
            hedge_prices,
            start_date=may_start,
            end_date=may_end,
            base_values=may_values,
            segment_name='2026-05'
        )
        if not df_may.empty:
            segments.append(df_may)

    # June: sell CPSEETF (take its latest Buy_Hold_Value from May), deploy into PHARMABEES using PHARMABEES OPEN
    june_start = pd.Timestamp('2026-06-01')
    df_june = pd.DataFrame()
    if portfolio_end_date >= june_start:
        # prefer CPSEETF value from May snapshot
        cpse_value = 0.0
        source_snapshot = None
        if 'df_may' in locals() and not df_may.empty and 'CPSEETF' in df_may['Ticker'].unique():
            snap_date = df_may['Date'].max()
            source_snapshot = df_may[df_may['Date'] == snap_date].set_index('Ticker')['Buy_Hold_Value'].to_dict()
            cpse_value = source_snapshot.get('CPSEETF', 0.0)
        elif 'df_mar' in locals() and not df_mar.empty and 'CPSEETF' in df_mar['Ticker'].unique():
            snap_date = df_mar['Date'].max()
            source_snapshot = df_mar[df_mar['Date'] == snap_date].set_index('Ticker')['Buy_Hold_Value'].to_dict()
            cpse_value = source_snapshot.get('CPSEETF', 0.0)

        if cpse_value > 0.0:
            # base allocation: take source_snapshot but replace CPSEETF with PHARMABEES
            june_values = {k: v for k, v in source_snapshot.items() if k != 'CPSEETF'}
            june_values['PHARMABEES'] = june_values.get('PHARMABEES', 0.0) + cpse_value

            df_june = build_weighted_hedge_segment(
                hedge_prices,
                start_date=june_start,
                end_date=portfolio_end_date,
                base_values=june_values,
                segment_name='2026-06_onward',
                new_tickers=['PHARMABEES']  # PHARMABEES bought fresh at Open on June 1
            )

            if not df_june.empty:
                # For PHARMABEES use Open at june_start (first available Open >= june_start)
                hc_prices = hedge_prices[(hedge_prices['Ticker'] == 'PHARMABEES') & (hedge_prices['Date'] >= june_start)].sort_values('Date')
                if not hc_prices.empty:
                    hc_open = float(hc_prices.iloc[0]['Open'])
                else:
                    hc_open = None

                hc_mask = df_june['Ticker'] == 'PHARMABEES'
                if hc_mask.any() and hc_open:
                    df_june.loc[hc_mask, 'Buy_Price'] = hc_open
                    df_june.loc[hc_mask, 'Quantity'] = np.where(hc_open > 0, df_june.loc[hc_mask, 'Initial_Allocation'] / hc_open, 0.0)
                    # recompute Buy_Hold_Value for PHARMABEES rows from Initial_Allocation
                    df_june.loc[hc_mask, 'Buy_Hold_Value'] = df_june.loc[hc_mask, 'Initial_Allocation'] * (
                        (1 + df_june.loc[hc_mask, '%change'].fillna(0)).groupby(df_june.loc[hc_mask, 'Ticker']).cumprod()
                    )

                # Ensure CPSEETF rows are not present in june segment
                df_june = df_june[df_june['Ticker'] != 'CPSEETF']

                segments.append(df_june)

    if not segments:
        return pd.DataFrame()

    hedge_book = pd.concat(segments, ignore_index=True).sort_values(['Date', 'Ticker']).reset_index(drop=True)
    return hedge_book


In [4]:
import os
import pandas as pd

def prepare_and_process_portfolio(input_file, start_date, end_date, output_folder,
                                  process_portfolio,
                                  equity_allocation=75, gold_allocation=25):
    """
    Prepare portfolio dataframe with momentum stocks + GOLDBEES and process performance.
    """

    nav_df_raw = pd.read_excel(input_file).rename(columns={'End_Date': 'Date'})
    nav_df_raw['Date'] = pd.to_datetime(nav_df_raw['Date'])

    selected_cols = ['Date', 'Ticker']
    if 'Real_Rank' in nav_df_raw.columns:
        selected_cols.append('Real_Rank')

    nav_df = (
        nav_df_raw[(nav_df_raw['Date'] >= start_date) & (nav_df_raw['Date'] <= end_date)]
        .reset_index(drop=True)[selected_cols]
    )
    nav_df['Year-Month'] = nav_df['Date'].dt.to_period('M').astype(str)

    goldbees_df = pd.DataFrame({
        'Date': nav_df['Date'].drop_duplicates().sort_values(),
        'Ticker': 'GOLDBEES'
    })
    if 'Real_Rank' in nav_df.columns:
        goldbees_df['Real_Rank'] = np.nan
    goldbees_df['Year-Month'] = pd.to_datetime(goldbees_df['Date']).dt.to_period('M').astype(str)

    concat_df = (
        pd.concat([nav_df, goldbees_df], ignore_index=True)
          .sort_values(['Date', 'Ticker'])
          .reset_index(drop=True)
    )

    ticker_df = concat_df.query("Ticker != 'GOLDBEES'")
    symbol_list = ticker_df['Ticker'].unique()
    ticker_data_other_stocks, errors_other = fetch_truedata_history(
        ticker_list=symbol_list,
        duration='10 Y',
        bar_size='EOD',
        sleep_time=0.1
    )
    if errors_other:
        logging.warning(f'Failed to fetch data for {errors_other}')

    gold_df = concat_df.query("Ticker == 'GOLDBEES'")
    symbol_list = gold_df['Ticker'].unique()
    ticker_data_gold, errors_gold = fetch_truedata_history(
        ticker_list=symbol_list,
        duration='10 Y',
        bar_size='EOD',
        sleep_time=0.1
    )
    if errors_gold:
        logging.warning(f'Failed to fetch gold data for {errors_gold}')

    inception_date = pd.to_datetime(start_date)
    final_df_other_stocks = process_portfolio(
        ticker_df,
        ticker_data_other_stocks,
        equity_allocation,
        inception_date=inception_date
    )
    final_df_gold = process_portfolio(
        gold_df,
        ticker_data_gold,
        gold_allocation,
        inception_date=inception_date
    )

    final_df = (
        pd.concat([final_df_other_stocks, final_df_gold], ignore_index=True)
          .sort_values(['Date', 'Ticker'])
          .reset_index(drop=True)
    )

    if not os.path.exists(output_folder):
        os.makedirs(output_folder)

    middle_folder = os.path.basename(os.path.dirname(input_file))
    output_file = os.path.join(output_folder, f"{middle_folder}_gold_buy&hold_returns.xlsx")
    print(f"Final output path: {output_file}")

    return final_df

In [5]:
final_df = prepare_and_process_portfolio(
    input_file="C:\\Users\\anike\\Desktop\\Ocean_dev\\Momentum Handover\\Momentum Handover\\MOMENTUM_DB_2\\Stocks_old\\Nifty_500_2025_Apr_20_stocks_results\\master_momentum_summary.xlsx",
    start_date="2023-04-01",
    end_date=date.today().strftime('%Y-%m-%d'),
    output_folder="Trials",
    process_portfolio=process_portfolio
)

final_df

Connecting to TrueData...


(2026-06-19 09:29:12,699) WARNING :: Connected successfully to TrueData Historical Data Service...  (PID:28612 Thread:29440)
2026-06-19 09:29:12,699 - WARNING - Connected successfully to TrueData Historical Data Service... 
2026-06-19 09:29:12,701 - INFO - Connected via truedata.TD_hist


Connected.



2026-06-19 09:29:13,137 - INFO - Fetched data for ABCAPITAL (2180 rows).
2026-06-19 09:29:13,581 - INFO - Fetched data for ANANDRATHI (1120 rows).
2026-06-19 09:29:14,031 - INFO - Fetched data for ANANTRAJ (2477 rows).
2026-06-19 09:29:14,476 - INFO - Fetched data for BANKBARODA (2477 rows).
2026-06-19 09:29:14,934 - INFO - Fetched data for BANKINDIA (2477 rows).
2026-06-19 09:29:15,433 - INFO - Fetched data for CANBK (2477 rows).
2026-06-19 09:29:15,908 - INFO - Fetched data for CUMMINSIND (2477 rows).
2026-06-19 09:29:16,336 - INFO - Fetched data for FINCABLES (2477 rows).
2026-06-19 09:29:16,793 - INFO - Fetched data for GODFRYPHLP (2477 rows).
2026-06-19 09:29:17,266 - INFO - Fetched data for J&KBANK (2477 rows).
2026-06-19 09:29:17,741 - INFO - Fetched data for JINDALSAW (2477 rows).
2026-06-19 09:29:18,239 - INFO - Fetched data for JSL (2477 rows).
2026-06-19 09:29:18,699 - INFO - Fetched data for KAYNES (887 rows).
2026-06-19 09:29:19,225 - INFO - Fetched data for KIRLOSENG (247

Final output path: Trials\Nifty_500_2025_Apr_20_stocks_results_gold_buy&hold_returns.xlsx


,Date,Open,High,Low,Close,volume,oi,Ticker,%change,Initial_Allocation,Selection_Date,Buy_Hold_Value,Buy_Price,Quantity,Real_Rank,Total_Portfolio_Value
0,2023-04-03,154.00,154.90,152.00,153.80,2347046,0,ABCAPITAL,-0.001299,3.750000,2023-04-01,3.745130,153.80,0.024382,11.0,75.402342
1,2023-04-03,203.50,205.50,201.55,202.55,225128,0,ANANDRATHI,-0.004668,3.750000,2023-04-01,3.732494,202.55,0.018514,15.0,75.402342
2,2023-04-03,123.75,128.00,122.50,125.65,2365577,0,ANANTRAJ,0.015354,3.750000,2023-04-01,3.807576,125.65,0.029845,19.0,75.402342
3,2023-04-03,169.10,170.25,168.10,169.20,16659059,0,BANKBARODA,0.000591,3.750000,2023-04-01,3.752218,169.20,0.022163,7.0,75.402342
4,2023-04-03,75.00,76.45,74.10,75.95,8102076,0,BANKINDIA,0.012667,3.750000,2023-04-01,3.797500,75.95,0.049375,3.0,75.402342
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16711,2026-06-19,2804.00,2804.00,2776.80,2792.00,296287,0,MCX,-0.008945,12.773268,2026-06-01,12.070727,2890.50,0.004452,15.0,209.866294
16712,2026-06-19,365.90,370.25,362.40,369.75,777837,0,NATIONALUM,0.005438,15.937219,2026-06-01,13.883347,434.20,0.037591,3.0,209.866294
16713,2026-06-19,7345.00,7380.00,7298.50,7323.50,17850,0,NAVINFLUOR,0.002738,8.621715,2026-06-01,8.861291,6998.50,0.001232,18.0,209.866294
16714,2026-06-19,181.40,181.80,179.60,181.00,1412159,0,SAIL,-0.006368,10.206759,2026-06-01,9.039602,203.69,0.049701,6.0,209.866294


In [6]:
old_df = final_df[~((final_df['Date']>'2025-11-30') & (final_df['Ticker']=='GOLDBEES'))]
old_df



,Date,Open,High,Low,Close,volume,oi,Ticker,%change,Initial_Allocation,Selection_Date,Buy_Hold_Value,Buy_Price,Quantity,Real_Rank,Total_Portfolio_Value
0,2023-04-03,154.00,154.90,152.00,153.80,2347046,0,ABCAPITAL,-0.001299,3.750000,2023-04-01,3.745130,153.80,0.024382,11.0,75.402342
1,2023-04-03,203.50,205.50,201.55,202.55,225128,0,ANANDRATHI,-0.004668,3.750000,2023-04-01,3.732494,202.55,0.018514,15.0,75.402342
2,2023-04-03,123.75,128.00,122.50,125.65,2365577,0,ANANTRAJ,0.015354,3.750000,2023-04-01,3.807576,125.65,0.029845,19.0,75.402342
3,2023-04-03,169.10,170.25,168.10,169.20,16659059,0,BANKBARODA,0.000591,3.750000,2023-04-01,3.752218,169.20,0.022163,7.0,75.402342
4,2023-04-03,75.00,76.45,74.10,75.95,8102076,0,BANKINDIA,0.012667,3.750000,2023-04-01,3.797500,75.95,0.049375,3.0,75.402342
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16711,2026-06-19,2804.00,2804.00,2776.80,2792.00,296287,0,MCX,-0.008945,12.773268,2026-06-01,12.070727,2890.50,0.004452,15.0,209.866294
16712,2026-06-19,365.90,370.25,362.40,369.75,777837,0,NATIONALUM,0.005438,15.937219,2026-06-01,13.883347,434.20,0.037591,3.0,209.866294
16713,2026-06-19,7345.00,7380.00,7298.50,7323.50,17850,0,NAVINFLUOR,0.002738,8.621715,2026-06-01,8.861291,6998.50,0.001232,18.0,209.866294
16714,2026-06-19,181.40,181.80,179.60,181.00,1412159,0,SAIL,-0.006368,10.206759,2026-06-01,9.039602,203.69,0.049701,6.0,209.866294


In [7]:
np.sort(old_df['Ticker'].unique())



array(['AAVAS', 'ABB', 'ABCAPITAL', 'ABSLAMC', 'ACE', 'ACUTAAS',
       'ADANIENSOL', 'ADANIGREEN', 'ADANIPORTS', 'AIIL', 'AJANTPHARM',
       'ALIVUS', 'AMBER', 'ANANDRATHI', 'ANANTRAJ', 'ANGELONE',
       'APLAPOLLO', 'ASHOKLEY', 'ASTERDM', 'AUBANK', 'AUROPHARMA',
       'AXISBANK', 'BAJFINANCE', 'BANKBARODA', 'BANKINDIA', 'BASF', 'BDL',
       'BEL', 'BEML', 'BERGEPAINT', 'BHARATFORG', 'BHARTIARTL',
       'BHARTIHEXA', 'BHEL', 'BLUESTARCO', 'BPCL', 'BSE', 'BSOFT',
       'CANBK', 'CGPOWER', 'CHAMBLFERT', 'CHENNPETRO', 'CHOLAFIN',
       'CHOLAHLDNG', 'COALINDIA', 'COCHINSHIP', 'COFORGE', 'COHANCE',
       'COLPAL', 'CONCORDBIO', 'COROMANDEL', 'CUB', 'CUMMINSIND',
       'CYIENT', 'DATAPATTNS', 'DBREALTY', 'DEEPAKFERT', 'DELHIVERY',
       'DIVISLAB', 'DIXON', 'DLF', 'DOMS', 'ECLERX', 'EICHERMOT',
       'EIHOTEL', 'ELECON', 'EMAMILTD', 'EMMVEE', 'ENGINERSIN', 'ERIS',
       'ETERNAL', 'EXIDEIND', 'FEDERALBNK', 'FINCABLES', 'FLUOROCHEM',
       'FORTIS', 'GESHIP', 'GLAND', 'GLENMARK

In [8]:
# Build hedge book with the same monthly rebalancing rules used in the script
portfolio_end_date = pd.to_datetime(final_df['Date']).max()
df = build_rebalanced_hedge_book(
    base_portfolio_df=final_df,
    portfolio_end_date=portfolio_end_date,
    default_hedge_value=25.0
 )
df

2026-06-19 09:31:38,323 - INFO - Fetched data for GOLDBEES (1240 rows).
2026-06-19 09:31:38,934 - INFO - Fetched data for SILVERBEES (1082 rows).
2026-06-19 09:31:39,565 - INFO - Fetched data for MOGSEC (1241 rows).
2026-06-19 09:31:40,174 - INFO - Fetched data for LIQUIDCASE (595 rows).
2026-06-19 09:31:40,799 - INFO - Fetched data for CPSEETF (1240 rows).
2026-06-19 09:31:41,440 - INFO - Fetched data for PHARMABEES (1228 rows).
2026-06-19 09:31:42,036 - INFO - Fetched data for NEXT50IETF (619 rows).


,Date,Ticker,Open,Close,%change,Initial_Allocation,Buy_Hold_Value,Buy_Price,Quantity,Selection_Date,Hedge_Segment,Total_Portfolio_Value
0,2025-12-01,GOLDBEES,106.09,106.72,0.020073,30.952663,31.573964,106.72,0.290036,2025-12-01,2025-12_to_2026-01,52.807408
1,2025-12-01,MOGSEC,62.70,62.82,-0.002224,10.317554,10.294612,62.82,0.164240,2025-12-01,2025-12_to_2026-01,52.807408
2,2025-12-01,SILVERBEES,166.02,166.21,0.060216,10.317554,10.938832,166.21,0.062075,2025-12-01,2025-12_to_2026-01,52.807408
3,2025-12-02,GOLDBEES,106.65,105.63,-0.010214,30.952663,31.251479,106.72,0.290036,2025-12-01,2025-12_to_2026-01,52.440775
4,2025-12-02,MOGSEC,62.97,62.90,0.001273,10.317554,10.307722,62.82,0.164240,2025-12-01,2025-12_to_2026-01,52.440775
...,...,...,...,...,...,...,...,...,...,...,...,...
377,2026-06-18,PHARMABEES,25.03,24.90,0.006061,12.636375,12.431676,25.31,0.499264,2026-06-01,2026-06_onward,66.871398
378,2026-06-19,GOLDBEES,119.85,119.18,-0.024075,14.035968,13.002773,127.49,0.110095,2026-06-01,2026-06_onward,66.583874
379,2026-06-19,LIQUIDCASE,114.62,114.62,0.000262,27.032243,27.107924,114.33,0.236441,2026-06-01,2026-06_onward,66.583874
380,2026-06-19,NEXT50IETF,75.31,76.47,-0.001697,13.720782,13.991575,73.78,0.185969,2026-06-01,2026-06_onward,66.583874


In [9]:
# Hedge dataframe is already built in previous cell
df[['Date', 'Ticker', 'Buy_Hold_Value']].head()

,Date,Ticker,Buy_Hold_Value
0,2025-12-01,GOLDBEES,31.573964
1,2025-12-01,MOGSEC,10.294612
2,2025-12-01,SILVERBEES,10.938832
3,2025-12-02,GOLDBEES,31.251479
4,2025-12-02,MOGSEC,10.307722


In [10]:
# No-op: valuation already computed in hedge builder cell
df.tail()

,Date,Ticker,Open,Close,%change,Initial_Allocation,Buy_Hold_Value,Buy_Price,Quantity,Selection_Date,Hedge_Segment,Total_Portfolio_Value
377,2026-06-18,PHARMABEES,25.03,24.90,0.006061,12.636375,12.431676,25.31,0.499264,2026-06-01,2026-06_onward,66.871398
378,2026-06-19,GOLDBEES,119.85,119.18,-0.024075,14.035968,13.002773,127.49,0.110095,2026-06-01,2026-06_onward,66.583874
379,2026-06-19,LIQUIDCASE,114.62,114.62,0.000262,27.032243,27.107924,114.33,0.236441,2026-06-01,2026-06_onward,66.583874
380,2026-06-19,NEXT50IETF,75.31,76.47,-0.001697,13.720782,13.991575,73.78,0.185969,2026-06-01,2026-06_onward,66.583874
381,2026-06-19,PHARMABEES,24.40,25.00,0.004016,12.636375,12.481603,25.31,0.499264,2026-06-01,2026-06_onward,66.583874


In [11]:
conc_df = pd.concat([old_df, df])
conc_df



,Date,Open,High,Low,Close,volume,oi,Ticker,%change,Initial_Allocation,Selection_Date,Buy_Hold_Value,Buy_Price,Quantity,Real_Rank,Total_Portfolio_Value,Hedge_Segment
0,2023-04-03,154.00,154.90,152.00,153.80,2347046.0,0.0,ABCAPITAL,-0.001299,3.750000,2023-04-01,3.745130,153.80,0.024382,11.0,75.402342,NaN
1,2023-04-03,203.50,205.50,201.55,202.55,225128.0,0.0,ANANDRATHI,-0.004668,3.750000,2023-04-01,3.732494,202.55,0.018514,15.0,75.402342,NaN
2,2023-04-03,123.75,128.00,122.50,125.65,2365577.0,0.0,ANANTRAJ,0.015354,3.750000,2023-04-01,3.807576,125.65,0.029845,19.0,75.402342,NaN
3,2023-04-03,169.10,170.25,168.10,169.20,16659059.0,0.0,BANKBARODA,0.000591,3.750000,2023-04-01,3.752218,169.20,0.022163,7.0,75.402342,NaN
4,2023-04-03,75.00,76.45,74.10,75.95,8102076.0,0.0,BANKINDIA,0.012667,3.750000,2023-04-01,3.797500,75.95,0.049375,3.0,75.402342,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
377,2026-06-18,25.03,NaN,NaN,24.90,NaN,NaN,PHARMABEES,0.006061,12.636375,2026-06-01,12.431676,25.31,0.499264,NaN,66.871398,2026-06_onward
378,2026-06-19,119.85,NaN,NaN,119.18,NaN,NaN,GOLDBEES,-0.024075,14.035968,2026-06-01,13.002773,127.49,0.110095,NaN,66.583874,2026-06_onward
379,2026-06-19,114.62,NaN,NaN,114.62,NaN,NaN,LIQUIDCASE,0.000262,27.032243,2026-06-01,27.107924,114.33,0.236441,NaN,66.583874,2026-06_onward
380,2026-06-19,75.31,NaN,NaN,76.47,NaN,NaN,NEXT50IETF,-0.001697,13.720782,2026-06-01,13.991575,73.78,0.185969,NaN,66.583874,2026-06_onward


In [12]:
import plotly.express as px

# âœ… Group by Date and calculate total portfolio value
portfolio_summary = (
    conc_df.groupby("Date", as_index=False)["Buy_Hold_Value"].sum()
)
# âœ… Plot with Plotly
fig = px.line(
    portfolio_summary,
    x="Date",
    y="Buy_Hold_Value",
    title="Buy_Hold_Value Over Time",
    labels={"Date": "Date", "Buy_Hold_Value": "Buy_Hold_Value"},
    markers=True
)

fig.update_traces(line=dict(width=2))
fig.update_layout(width=1000,   # ðŸ”‘ width
                  height=500)    # ðŸ”‘ height
fig.show()



In [13]:
# Momentum/Automating Momentum True Data/Trials/Nifty_500_2025_Apr_20_stocks_results_GoldSilverDebt_buy&hold_returns.xlsx



In [15]:
conc_df.to_excel('C:\\Users\\anike\\Desktop\\Ocean_dev\\Momentum Handover\\Momentum Handover\\Trials\\Nifty_500_2025_Apr_20_stocks_results_GoldSilverDebt_buy&hold_returns.xlsx', index=False)
# \Trials



In [16]:
nse = fetch_truedata_history(
    ticker_list = ['NIFTY 500'],
    duration = '5 Y',
    bar_size = 'EOD',
    sleep_time= 0.1
)[0]
nse = nse[["Date", "Close"]].rename(columns={'Close':'Buy_Hold_Value'})
nse['%change'] = nse['Buy_Hold_Value'].pct_change()
nse = nse[nse['Date'] >= '2023-04-01']
nse.to_excel(r'C:\Users\anike\Desktop\Ocean_dev\Momentum Handover\Momentum Handover\Trials\nse500_Nifty_500_2025_Apr_nse500_nse500_nse500_nse500_nse500_returns.xlsx', index=False)
nse



2026-06-19 09:32:17,502 - INFO - Fetched data for NIFTY 500 (1241 rows).


,Date,Buy_Hold_Value,%change
445,2023-04-03,14601.95,0.003029
446,2023-04-05,14709.40,0.007359
447,2023-04-06,14759.20,0.003386
448,2023-04-10,14790.55,0.002124
449,2023-04-11,14867.25,0.005186
...,...,...,...
1236,2026-06-15,22891.30,0.012898
1237,2026-06-16,22997.25,0.004628
1238,2026-06-17,23109.70,0.004890
1239,2026-06-18,23206.45,0.004187
